# 1693. Daily Leads and Partners

## Problem
DailySales(date_id,make_name,lead_id,partner_id): count distinct lead_id and partner_id for every date and make. Return date_id,make_name,unique_leads,unique_partners.

[Doocs problem reference](https://github.com/doocs/leetcode/blob/main/solution/1600-1699/1693.Daily%20Leads%20and%20Partners/README_EN.md) · [LeetCode](https://leetcode.com/problems/daily-leads-and-partners/)

[Community solution index (Silversaya, 2023)](https://leetcode.cn/discuss/post/3573106/zhe-shi-pandas30tian-ji-hua-de-ti-jie-qu-v0wv/) · [LeetCode discussions](https://leetcode.com/problems/daily-leads-and-partners/solutions/)

Community links are for further reading; the implementations below are independently written.

## Practice first
Run the data cell and try the blank cell before reading solutions.

In [ ]:
import pandas as pd
from pandas.testing import assert_frame_equal

daily_sales = pd.DataFrame([["2024-01-01","A",1,10],["2024-01-01","A",1,11],["2024-01-01","A",2,11],["2024-01-01","B",1,10]], columns=["date_id","make_name","lead_id","partner_id"])
expected = pd.DataFrame([["2024-01-01","A",2,2],["2024-01-01","B",1,1]], columns=["date_id","make_name","unique_leads","unique_partners"])
display(daily_sales)
display(expected)

In [ ]:
# Your solution here
# result = ...

## Solutions

### Solution 1 — Named aggregation
不同欄位各自 distinct count。

In [ ]:
def solution_1(daily_sales):
    return daily_sales.groupby(['date_id','make_name'],as_index=False).agg(unique_leads=('lead_id','nunique'),unique_partners=('partner_id','nunique'))

result_1 = solution_1(daily_sales.copy())
display(result_1)

### Solution 2 — Multi-column nunique
同一 aggregation 套用多個欄位。

In [ ]:
def solution_2(daily_sales):
    return daily_sales.groupby(['date_id','make_name'])[['lead_id','partner_id']].nunique().rename(columns={'lead_id':'unique_leads','partner_id':'unique_partners'}).reset_index()

result_2 = solution_2(daily_sales.copy())
display(result_2)

### Solution 3 — Separate deduplication
依兩套粒度去重後依 MultiIndex 對齊；題目 IDs 非空。

In [ ]:
def solution_3(daily_sales):
    keys = ['date_id','make_name']
    l = daily_sales.drop_duplicates(keys+['lead_id']).groupby(keys).size().rename('unique_leads')
    p = daily_sales.drop_duplicates(keys+['partner_id']).groupby(keys).size().rename('unique_partners')
    return pd.concat([l,p],axis=1).reset_index()

result_3 = solution_3(daily_sales.copy())
display(result_3)

## Check all solutions
Rows are normalized only for the example comparison; preserve required ordering in the solutions themselves.

In [ ]:
def normalize(df):
    return df.sort_values(list(df.columns),na_position='last').reset_index(drop=True)
assert_frame_equal(normalize(result_1), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_2), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_3), normalize(expected), check_dtype=False)
print('All example checks passed')

## Review
Compare row grain, missing matches, duplicate keys and zero-count categories.